# Currency and timeliness of the hotel bookings dataset

**Data source:** Antonio, N., de Almeida, A. and Nunes, L. (2019). *Hotel booking demand datasets*. Data in Brief, 22, 41–49. https://doi.org/10.1016/j.dib.2018.11.126 (file `hotel_bookings_dataset.csv`).
**Date:** 2026-11-05
**Authors:** Group 3, Presentation and Visualization (Infovis2627)
**Description:** This notebook answers three questions of the brief with evidence from the data (Part 1, task 1.5): how current the data is, how often it should be updated, and whether it is real or forecast.

---

## Contents

1. [Data loading](#1.-Data-loading)
2. [Initial exploration: time covered](#2.-Initial-exploration:-time-covered)
3. [Cleaning and preparation](#3.-Cleaning-and-preparation)
4. [Analysis: real or forecast, and age of the data](#4.-Analysis:-real-or-forecast,-and-age-of-the-data)
5. [Accessible visualisation](#5.-Accessible-visualisation)
6. [Export](#6.-Export)
7. [Conclusions](#Conclusions)

In [1]:
# ── Imports ──────────────────────────────────────────────────────────────────
import sys
from pathlib import Path
import pandas as pd

RAW_FILE   = Path('../../Course information/hotel_bookings_dataset.csv')
OUTPUT_DIR = Path('currency_timeliness_output')
OUTPUT_DIR.mkdir(exist_ok=True)

# Date of this analysis, used only to measure the age of the data
ANALYSIS_DATE = pd.Timestamp('2026-11-05')
print('Python', sys.version.split()[0], '| pandas', pd.__version__, '| analysis date', ANALYSIS_DATE.date())

Python 3.13.18 | pandas 3.0.5 | analysis date 2026-11-05


## 1. Data loading

In [2]:
# ── Load and build the dates ─────────────────────────────────────────────────
bookings = pd.read_csv(RAW_FILE, keep_default_na=False, na_values=['NA', 'NULL'])

month_number = {'January': 1, 'February': 2, 'March': 3, 'April': 4, 'May': 5, 'June': 6,
                'July': 7, 'August': 8, 'September': 9, 'October': 10, 'November': 11, 'December': 12}
arrival_date = pd.to_datetime(pd.DataFrame({'year': bookings['arrival_date_year'],
                                            'month': bookings['arrival_date_month'].map(month_number),
                                            'day': bookings['arrival_date_day_of_month']}))
status_date = pd.to_datetime(bookings['reservation_status_date'])

# Booking date = arrival date minus lead time (the paper defines lead time this way)
booking_date = arrival_date - pd.to_timedelta(bookings['lead_time'], unit='D')

print(f'Rows: {len(bookings):,}')
pd.DataFrame({'arrival': arrival_date, 'status': status_date, 'booking (derived)': booking_date}).agg(['min', 'max'])

Rows: 119,390


,arrival,status,booking (derived)
min,2015-07-01,2014-10-17,2013-06-24
max,2017-08-31,2017-09-14,2017-08-31


## 2. Initial exploration: time covered

In [3]:
# ── Months of arrival covered in each year ───────────────────────────────────
months_per_year = (pd.DataFrame({'year': arrival_date.dt.year, 'month': arrival_date.dt.month})
                   .groupby('year')['month'].agg(['min', 'max', 'nunique'])
                   .rename(columns={'min': 'first month', 'max': 'last month', 'nunique': 'months covered'}))
months_per_year['bookings'] = arrival_date.dt.year.value_counts().sort_index()
months_per_year

,first month,last month,months covered,bookings
year,,,,
2015,7,12,6,21996
2016,1,12,12,56707
2017,1,8,8,40687


**Finding:** the arrivals cover 26 consecutive months, from 1 July 2015 to 31 August 2017. Only **2016 is a complete year**; 2015 has its last six months and 2017 its first eight. Totals per year are therefore not comparable (2016 would always look biggest), and any comparison between years must use the same months. The oldest booking was made on 24 June 2013 (arrival date minus lead time).

## 3. Cleaning and preparation

Nothing is cleaned; the booking date is derived only to describe the time span.

## 4. Analysis: real or forecast, and age of the data

In [4]:
# ── Is every booking closed? ─────────────────────────────────────────────────

# A file of future bookings ("on the books") would contain bookings still waiting for their arrival.
# Here every booking should have a final status, and the last status date should match the last departure
print('Reservation statuses present:', sorted(bookings['reservation_status'].unique()))
last_departure = (arrival_date + pd.to_timedelta(bookings['stays_in_weekend_nights'] + bookings['stays_in_week_nights'],
                                                 unit='D')).max()
print('Last departure date:', last_departure.date(), '| last status date:', status_date.max().date())

Reservation statuses present: ['Canceled', 'Check-Out', 'No-Show']
Last departure date: 2017-09-14 | last status date: 2017-09-14


In [5]:
# ── Age of the data ───────────────────────────────────────────────────────────
age_years = (ANALYSIS_DATE - status_date.max()).days / 365.25
print(f'Time between the last recorded status and this analysis: {age_years:.1f} years')

Time between the last recorded status and this analysis: 9.1 years


### Findings

**Real or forecast.** The data is **real, historical** data. The source paper says it was extracted from the hotels' property management systems ("hotel real data"), and the file agrees: every booking has a final status (`Check-Out`, `Canceled` or `No-Show`), none is still pending, and the last status date (2017-09-14) is the day of the last departure. Nothing in it is a forecast or a target.

**Currency.** It is a **closed snapshot**: no booking arriving after August 2017 is included, and the source has never been updated. At the date of this analysis the newest record is about nine years old, so the dashboard shows patterns of 2015–2017 (before the COVID-19 pandemic changed travel), not the current situation. This must be visible in the dashboard next to the source.

**How often it should be updated.** As delivered, it cannot be updated: it is a published research dataset. If the same dashboard were connected to a hotel's live system, the right frequency would depend on its use, which Part 2 will decide: bookings and cancellations change every day, so an operational view would need a **daily** update, while the seasonal patterns found in task 1.4 change over months, so a strategic view would only need a **monthly** update, after each month closes.

## 5. Accessible visualisation

No new chart is needed: the time covered is shown by the chart "Bookings per month of arrival in each hotel" of the relationships notebook (`eda_bookings_over_time.png`), and the table above gives the exact months.

## 6. Export

In [6]:
# ── Export ────────────────────────────────────────────────────────────────────
months_per_year.to_csv(OUTPUT_DIR / 'months_covered_per_year.csv')
pd.Series({'first arrival': arrival_date.min().date(), 'last arrival': arrival_date.max().date(),
           'first booking (derived)': booking_date.min().date(), 'last status date': status_date.max().date(),
           'last departure': last_departure.date(), 'age at analysis (years)': round(age_years, 1)},
          name='value').to_csv(OUTPUT_DIR / 'time_span.csv')
sorted(p.name for p in OUTPUT_DIR.iterdir())

['months_covered_per_year.csv', 'time_span.csv']

## Conclusions

### Main findings
1. Real, historical bookings of two Portuguese hotels, with arrivals from July 2015 to August 2017; no forecast values.
2. A closed snapshot, about nine years old at the time of this analysis; only 2016 is a complete year.
3. Update frequency: none for this dataset; for a live version, daily for operational use or monthly for strategic use (decided with the audience in Part 2).

### Limitations
- The extraction date is not stated in the paper; it is on or after 2017-09-14, the last status date.

### Suggested next steps
- Task 1.6: cleaning rules.